# 🔄 Módulo 13 — Workflows: Ejecutores (Executors)

> **Objetivo:** Introducir el concepto de workflow — un grafo de pasos (ejecutores) que procesan datos.

## ¿Qué es un workflow?

Un workflow es un **grafo dirigido de ejecutores**. Cada ejecutor:
- Recibe un mensaje tipado
- Hace algo con él (transformarlo, calcular, llamar a un servicio, etc.)
- O bien lo reenvía al siguiente nodo (`ctx.send_message`), o emite la salida final del workflow (`ctx.yield_output`)

```
input ─→ [Ejecutor A] ─→ [Ejecutor B] ─→ [Ejecutor C] ─→ output
```

## Diferencia vs. agentes

| Aspecto | Agentes (Módulos 00-08) | Workflows (Módulos 09-11) |
|---------|-------------------------|----------------------------|
| Decisiones | El modelo LLM decide qué hacer | Tú defines el flujo explícitamente |
| Coste | Cada paso llama al LLM | Pasos puros Python = $0 |
| Predictibilidad | Probabilística | Determinística |
| Uso típico | Conversación, razonamiento | Pipelines de datos, orquestación |

## API

```python
from agent_framework import Executor, WorkflowBuilder, WorkflowContext, executor, handler

# Opción 1: clase
class MyExecutor(Executor):
    @handler
    async def my_handler(self, msg: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(msg.upper())

# Opción 2: función
@executor(id="my_func")
async def my_func(msg: str, ctx: WorkflowContext[Never, str]) -> None:
    await ctx.yield_output(f"Result: {msg}")

# Construir y ejecutar
workflow = WorkflowBuilder(start_executor=A).add_edge(A, B).build()
events = await workflow.run("input")
print(events.get_outputs())
```

> 🎉 **Estos módulos NO requieren Azure OpenAI** — todo se ejecuta localmente.


## ⚙️ Setup (no requiere ChatClient para este módulo)

In [ ]:
import sys
from pathlib import Path

_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from typing_extensions import Never
from agent_framework import (
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    handler,
)

print("✅ Listo - no necesitamos Azure OpenAI para este módulo")


## 1️⃣ Workflow secuencial con 2 ejecutores

Patrón clásico pipeline: cada ejecutor recibe la salida del anterior.

Ejemplo: `"hello world"` → Uppercase → `"HELLO WORLD"` → Reverse → `"DLROW OLLEH"`


In [ ]:
class UppercaseExecutor(Executor):
    def __init__(self, id: str = "uppercase"):
        super().__init__(id=id)

    @handler
    async def to_upper(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text.upper())


@executor(id="reverse_terminal")
async def reverse_terminal(text: str, ctx: WorkflowContext[Never, str]) -> None:
    # WorkflowContext[Never, str] = no envía downstream, sólo emite output final
    await ctx.yield_output(text[::-1])


uppercase = UppercaseExecutor()

workflow = (
    WorkflowBuilder(start_executor=uppercase)
    .add_edge(uppercase, reverse_terminal)
    .build()
)

events = await workflow.run("hello world")
print(f"✅ Workflow secuencial → {events.get_outputs()}")


## 2️⃣ Pipeline de 3 ejecutores

Encadenamos transformaciones: mayúsculas → invertir → contar palabras.

Observa cómo cada ejecutor declara su tipo de salida con `WorkflowContext[T]`.


In [ ]:
class ReverseExecutor(Executor):
    def __init__(self, id: str = "reverse"):
        super().__init__(id=id)

    @handler
    async def reverse(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text[::-1])


class WordCountExecutor(Executor):
    def __init__(self, id: str = "word_count"):
        super().__init__(id=id)

    @handler
    async def count(self, text: str, ctx: WorkflowContext[Never, str]) -> None:
        words = len([w for w in text.split(" ") if w])
        await ctx.yield_output(f"Words: {words}")


uc = UppercaseExecutor()
rv = ReverseExecutor()
wc = WordCountExecutor()

workflow = (
    WorkflowBuilder(start_executor=uc)
    .add_edge(uc, rv)
    .add_edge(rv, wc)
    .build()
)

events = await workflow.run("one two three")
print(f"✅ Pipeline 3 pasos → {events.get_outputs()}")


## 3️⃣ Estado compartido entre ejecutores

A veces dos ejecutores necesitan compartir datos sin pasarlos directamente como mensaje.
`ctx.set_shared_state(key, value)` y `ctx.get_shared_state(key)` son tu amigo aquí.


In [ ]:
import uuid


@executor(id="state_writer")
async def state_writer(text: str, ctx: WorkflowContext[str]) -> None:
    # Guarda en estado compartido y reenvía sólo la clave
    key = uuid.uuid4().hex
    ctx.set_state(key, text.upper())
    await ctx.send_message(key)


@executor(id="state_reader")
async def state_reader(key: str, ctx: WorkflowContext[Never, str]) -> None:
    # Lee el valor usando la clave recibida
    value = ctx.get_state(key)
    await ctx.yield_output(f"Read from state: {value}")


workflow = (
    WorkflowBuilder(start_executor=state_writer)
    .add_edge(state_writer, state_reader)
    .build()
)

events = await workflow.run("shared data")
print(f"✅ Estado compartido → {events.get_outputs()}")


## 🎯 Resumen

| Concepto | API |
|----------|-----|
| Ejecutor (clase) | `class X(Executor):` con método `@handler async def ...(self, msg, ctx)` |
| Ejecutor (función) | `@executor(id="...") async def fn(msg, ctx) -> None:` |
| Reenviar a downstream | `await ctx.send_message(value)` con `WorkflowContext[T]` |
| Emitir salida final | `await ctx.yield_output(value)` con `WorkflowContext[Never, T]` |
| Construir | `WorkflowBuilder(start_executor=A).add_edge(A, B).build()` |
| Ejecutar | `events = await workflow.run(input); events.get_outputs()` |
| Estado compartido | `ctx.set_shared_state(k, v)` / `await ctx.get_shared_state(k)` |

**Siguiente módulo →** [Módulo 14 — Workflows: Edges Condicionales](./14_workflows_edges.ipynb)
